# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Dataset Croissant schema URL:**  
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets and their @id
print("Available Record Sets (by @id):")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']}")

# If there are no record sets declared in the top-level, let's try to auto-detect them from the schema
if len(record_sets) == 0:
    print("\nNo recordSet declared explicitly in the top-level Croissant metadata. Attempting to auto-detect possible data tables from files...")
    # Try to use dataset.tables (added in mlcroissant >=v0.6)
    # Otherwise, print the available data distribution URLs
    if hasattr(dataset, 'tables') and len(dataset.tables) > 0:
        for tbl in dataset.tables:
            print(f"- Table @id: {tbl['@id'] if '@id' in tbl else '(no @id)'}  Name: {tbl.get('name', '(no name)')}")
    else:
        # Fallback: Show the distribution URLs
        print("Distribution URLs (could represent data tables):")
        for dist in getattr(metadata, 'distribution', []):
            print(f"- {dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist}")

### Examine fields within the tabular record set

Let's try to extract the available field `@id`s (column IDs) from the dataset. We'll inspect one of the detected data tables if available.

In [ ]:
# Try to inspect fields (columns) from the first table/record set if available
table_id = None
fields = []

if len(record_sets) > 0:
    table_id = record_sets[0]['@id']
    fields = record_sets[0].get('field', [])
elif hasattr(dataset, 'tables') and len(dataset.tables) > 0:
    table_id = dataset.tables[0]['@id'] if '@id' in dataset.tables[0] else None
    fields = dataset.tables[0].get('field', [])
else:
    print("No tabular data record set with fields could be detected explicitly in the package.")

print(f"\nFirst detected table/record set @id: {table_id}")
if fields:
    print("Fields in this table (by @id):")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
        print(f"- {field_id}")
else:
    print("No fields could be auto-detected for this table. We'll infer columns after loading the data.")

## 3. Data Extraction
Load data from the main tabular record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Since the schema does not explicitly define `recordSet` at top-level, we'll try to stream all records via the detected table's `@id` (if available), or default to showing how to load from the available tables.

In [ ]:
# Define chosen record set/table @id for extraction
if table_id is None:
    raise ValueError('No tabular record set detected. Aborting extraction!')

record_set_id = table_id

# Extract data records into a DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from record set '{record_set_id}'.")
print("\nColumn names (fields, by @id):")
print(list(df.columns))
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records on specific criteria, normalizing numeric fields, and grouping.

In [ ]:
# Select a numeric field for analysis via column (field) @id
# We'll search for a numeric field (e.g., 'Age' or similar) by checking data types
numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_field_candidates:
    # Attempt to auto-convert from string
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            continue
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

# Pick the first numeric field found
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Selected numeric field: {numeric_field_id}")
else:
    raise ValueError('No numeric fields detected for analysis!')

# Filtering: Set a threshold and filter
threshold = df[numeric_field_id].quantile(0.8)
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} ({len(filtered_df)} records):")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field
categorical_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]
group_field = None
if categorical_candidates:
    # Try avoid the index field, prefer something that looks like a group (e.g., 'Sex', 'Site', 'Subtype')
    for col in categorical_candidates:
        if any(k in col.lower() for k in ['sex', 'site', 'group', 'subtype']):
            group_field = col
            break
    if not group_field:
        group_field = categorical_candidates[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field}' (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable categorical column found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, e.g., using histograms and boxplots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.grid(True)
plt.show()

if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.grid(True, axis='y')
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, inspect, and perform initial exploration of a FAIR<sup>2</sup> schema dataset using the `mlcroissant` library. Using entity `@id`s for record sets and fields enables robust programmatic access. Further statistical analyses, clinical insights, and advanced modeling can be built on this workflow.
